# Bootstrap & Preflight — 09-Champion_AutoWire.ipynb

**What this does**
- Verifies expected upstream notebooks have been executed and essential artefact folders exist.
- Prints clear guidance to run prerequisites if required files/folders are missing.

**Upstream prerequisites (recommended order)**
- `01-Setup_Preflight.ipynb`
- `02-Feature_Engineering.ipynb`
- `03-Feature_Selection.ipynb`
- `04-Baseline_and_BO.ipynb`
- `06-Ensembles.ipynb`
- `07-CNN.ipynb`

**Checks performed**
- Confirms `DATA_PATH` exists (from Section 0.1).
- Ensures `staging/` and `out/` directories exist when required downstream.
- Provides actionable instructions when a check fails.


In [1]:
# ======================================================
# Bootstrap & Preflight — 09-Champion_AutoWire.ipynb
#   • Validates prerequisites and artefact folders
#   • Prints guidance if prerequisites are missing
# ======================================================
print(">>> Bootstrap & Preflight — 09-Champion_AutoWire.ipynb")
required = ['01-Setup_Preflight.ipynb', '02-Feature_Engineering.ipynb', '03-Feature_Selection.ipynb', '04-Baseline_and_BO.ipynb', '06-Ensembles.ipynb', '07-CNN.ipynb']
print("[bootstrap] Recommended upstream notebooks:", required)

# Check DATA_PATH existence if declared
if 'DATA_PATH' in globals():
    from pathlib import Path as _P
    dp = _P(DATA_PATH)
    if not dp.exists():
        print(f"[bootstrap][warn] DATA_PATH not found: {dp} — please verify in 01-Setup_Preflight (Section 0.1).")

# Check staging/out directories
from pathlib import Path as _P
if 'STAGE_ROOT' in globals():
    sr = _P(STAGE_ROOT); 
    if not sr.exists():
        print(f"[bootstrap][warn] STAGE_ROOT does not exist: {sr}. Run 01-Setup_Preflight end-to-end first.")
if 'OUT_ROOT' in globals():
    oroot = _P(OUT_ROOT);
    if not oroot.exists():
        print(f"[bootstrap][warn] OUT_ROOT does not exist: {oroot}. It will be created as needed, but prior steps may be required.")

# Feature artefacts helpful for downstream
from pathlib import Path as _P
feat_dir = _P('staging') / 'feat'
if not feat_dir.exists():
    print("[bootstrap][hint] 'staging/feat' not found — this notebook can generate it (Sections 3.3/3.4), or run 03-Feature_Selection first.")
else:
    mi_file = feat_dir / 'mi_series.csv'
    if not mi_file.exists():
        print("[bootstrap][hint] MI series not found at 'staging/feat/mi_series.csv' — run Section 3.3 to generate.")

print("[bootstrap] Preflight checks complete. Proceed with this notebook if no critical warnings above.")


>>> Bootstrap & Preflight — 09-Champion_AutoWire.ipynb
[bootstrap] Recommended upstream notebooks: ['01-Setup_Preflight.ipynb', '02-Feature_Engineering.ipynb', '03-Feature_Selection.ipynb', '04-Baseline_and_BO.ipynb', '06-Ensembles.ipynb', '07-CNN.ipynb']
[bootstrap] Preflight checks complete. Proceed with this notebook if no critical warnings above.


## Section 9 — Section

## Section 9.1 — Section

In [2]:
# =====================================================
# Section 9.1 — Champion Selection & Manifest
#   • Imports pathlib/json/time (fixes NameError)
#   • Loads metrics from out/metrics (best_tree, cnn) if no champion_obj provided
#   • Picks a champion, writes reports/champion.json
#   • Sets CHAMPION_AVAILABLE / CHAMPION_STATUS
# =====================================================
print(">>> Section 9.1 — Champion Selection: start")

from pathlib import Path
import json, time

# Canonicals/dirs
OUT_ROOT = globals().get("OUT_ROOT", "out")
out_dir = Path(OUT_ROOT)
reports_dir = out_dir / "reports"
models_dir  = out_dir / "models"
metrics_dir = out_dir / "metrics"
for d in (reports_dir, models_dir, metrics_dir):
    d.mkdir(parents=True, exist_ok=True)

# Upstream-provided (optional)
champion_obj   = globals().get("champion_obj", None)
champion_valid = bool(globals().get("champion_valid", False))

def _load_json(p: Path):
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            return None
    return None

def _pick_model_file(models_dir: Path, name_hint: str | None):
    # Try to find a model file containing the name hint
    if name_hint:
        exts = (".pkl", ".joblib", ".keras", ".onnx")
        cand = [p for p in models_dir.iterdir()
                if p.suffix.lower() in exts and name_hint.lower() in p.name.lower()]
        if cand:
            return cand[0].name
    # Fallback: first available
    all_models = [p for p in models_dir.iterdir() if p.suffix.lower() in (".pkl",".joblib",".keras",".onnx")]
    return all_models[0].name if all_models else None

if not champion_valid or champion_obj is None:
    best_tree = _load_json(metrics_dir / "best_tree.json") or {}
    cnn_meta  = _load_json(metrics_dir / "cnn.json") or {}

    tree_acc = best_tree.get("best_tree_val_acc", None)
    try: tree_acc = float(tree_acc) if tree_acc is not None else None
    except: tree_acc = None

    cnn_acc = cnn_meta.get("val_acc", globals().get("CNN_VAL_ACC", None))
    try: cnn_acc = float(cnn_acc) if cnn_acc is not None else None
    except: cnn_acc = None

    candidates = []
    if tree_acc is not None:
        candidates.append(("tree", tree_acc, best_tree.get("best_tree_name","tree")))
    if cnn_acc is not None:
        candidates.append(("cnn", cnn_acc, "cnn_benchmark"))

    if candidates:
        kind, val_acc, name = max(candidates, key=lambda x: x[1])
        champion_obj = {
            "status": "selected",
            "winner_kind": kind,
            "winner_name": name,
            "val_acc": val_acc,
            "model_file": _pick_model_file(models_dir, name if kind=="tree" else None),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        champion_valid = True
    else:
        champion_obj = {
            "status": "pending",
            "reason": "No model scores available for champion selection.",
            "required_next_steps": [
                "Section 4 — Train LightGBM (baseline or BO)",
                "Section 5.* — HPO variants (if used in selection)",
                "Section 6.1 — Ensemble metrics aggregation",
                "Section 2.1 — Payload sequence preprocessing (if CNN required)"
            ],
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        champion_valid = False

# Save champion file
champ_path = reports_dir / "champion.json"
champ_path.write_text(json.dumps(champion_obj, indent=2))
CHAMPION_AVAILABLE = champion_valid
CHAMPION_STATUS = champion_obj.get("status","pending")

print("[ok] champion.json saved →", champ_path if champion_valid else "[skip] champion pending")
print(">>> Section 9.1 — Champion Selection: complete")


>>> Section 9.1 — Champion Selection: start
[ok] champion.json saved → out/reports/champion.json
>>> Section 9.1 — Champion Selection: complete


## Section 9.2 — Section

## Section 9.3 — Section

In [3]:
# =====================================================
# Section 9.3 — Champion Audit via Cross-Validation
#   • Imports sklearn CV + metrics
#   • Wraps y safely into 1D array
#   • Runs quick CV on candidate models if available
# =====================================================
print(">>> Section 9.3 — Champion Audit: start")

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, make_scorer

def _safe_1d_y(y):
    if y is None:
        return np.array([], dtype=int)
    arr = np.asarray(getattr(y, "values", y))
    arr = np.atleast_1d(arr)
    return arr.ravel()

def _can_run_cv(X, y):
    try:
        n, m = X.shape
        return n > 0 and m > 0 and y is not None and len(y) == n
    except Exception:
        return False

# Expect X_train_t/y_train prepared earlier (e.g. from splits)
y_vec = _safe_1d_y(globals().get("y_train", None))

ok_run = _can_run_cv(globals().get("X_train_t", None), y_vec)
if not ok_run:
    print("[skip] CV not runnable — missing or mismatched X/y.")
else:
    models = {
        "lgbm_baseline": globals().get("baseline_model"),
        "xgboost": globals().get("xgb_model"),
        "manual_hpo": globals().get("manual_hpo_model"),
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True,
                         random_state=globals().get("RANDOM_STATE", 42))
    sc_ap = make_scorer(average_precision_score, needs_proba=True)
    summary = {}
    for name, model in models.items():
        if model is None:
            continue
        try:
            from sklearn.model_selection import cross_val_score
            scores = cross_val_score(model, globals()["X_train_t"], y_vec,
                                     cv=cv, scoring=sc_ap)
            summary[name] = float(np.mean(scores))
        except Exception as e:
            summary[name] = f"error: {e}"
    print(f"[9.3] CV summary: {summary}")

print(">>> Section 9.3 — Champion Audit: complete")


>>> Section 9.3 — Champion Audit: start


[skip] CV not runnable — missing or mismatched X/y.
>>> Section 9.3 — Champion Audit: complete


## Section 9.6 — Section

In [4]:
# =====================================================
# Section 9.6 — Champion Artefact Export
#   • Ensures reports/ and models/ directories exist
#   • Persists champion.json (copy) into reports/
#   • Prepares directory scaffolding for deployment
# =====================================================
print(">>> Section 9.6 — Champion Artefact Export: start")

from pathlib import Path
import shutil, json

# Canonical root
OUT_ROOT = globals().get("OUT_ROOT", "out")
out_dir = Path(OUT_ROOT)
reports_dir = out_dir / "reports"
models_dir  = out_dir / "models"
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# Champion object from earlier section
champion_obj = globals().get("champion_obj", None)
champion_path = reports_dir / "champion.json"

if champion_obj:
    champion_path.write_text(json.dumps(champion_obj, indent=2))
    print(f"[ok] wrote champion.json → {champion_path}")
else:
    print("[skip] no champion_obj defined — nothing to export.")

# (Optional) copy selected model file into models_dir
if champion_obj and champion_obj.get("model_file"):
    src_model = Path(OUT_ROOT) / "models" / champion_obj["model_file"]
    if src_model.exists():
        dst_model = models_dir / src_model.name
        if src_model.resolve() != dst_model.resolve():
            shutil.copy2(src_model, dst_model)
        print(f"[ok] ensured champion model available at {dst_model}")
    else:
        print(f"[warn] champion model file not found: {src_model}")

print(">>> Section 9.6 — Champion Artefact Export: complete")


>>> Section 9.6 — Champion Artefact Export: start
[ok] wrote champion.json → out/reports/champion.json
[ok] ensured champion model available at out/models/champion_model.joblib
>>> Section 9.6 — Champion Artefact Export: complete


## Section 9.7 — Section

In [5]:
# =====================================================
# Section 9.7 — Champion Drift Report
#   • Imports pandas/numpy
#   • Provides drift_report_from_batch() to analyse feature drift
# =====================================================
print(">>> Section 9.7 — Champion Drift Report: start")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib, json

stage = Path(globals().get("STAGE_ROOT", "staging"))

def _kl_divergence(exp_hist: np.ndarray, act_hist: np.ndarray, eps: float=1e-9) -> float:
    exp_pct = (exp_hist / max(exp_hist.sum(), 1)).astype(float) + eps
    act_pct = (act_hist / max(act_hist.sum(), 1)).astype(float) + eps
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

def drift_report_from_batch(df_raw: pd.DataFrame, max_features: int=100):
    """Generate simple drift report by comparing new batch distribution to training expectations."""
    preproc = None
    for p in [stage / "split_preproc" / "preprocessor.pkl", stage / "preprocessor.pkl"]:
        if p.exists():
            preproc = joblib.load(p)
            break
    if preproc is None:
        raise FileNotFoundError("[9.7] Preprocessor not found in staging.")

    # Transform raw df with fitted preprocessor
    X_new = preproc.transform(df_raw)
    feature_names = getattr(preproc, "get_feature_names_out", lambda: None)()
    if feature_names is None:
        feature_names = [f"f{i}" for i in range(X_new.shape[1])]

    # Compute histograms for first max_features
    drift = {}
    for i, fname in enumerate(feature_names[:max_features]):
        col = np.asarray(X_new[:, i]).ravel()
        hist_act, _ = np.histogram(col, bins=20, range=(col.min(), col.max()))
        # Load expected histogram if available
        exp_path = stage / "drift_baseline" / f"{fname}.npy"
        if exp_path.exists():
            hist_exp = np.load(exp_path)
            drift[fname] = _kl_divergence(hist_exp, hist_act)
        else:
            drift[fname] = None

    return drift

# Example usage (will skip if no data)
try:
    dummy = pd.DataFrame()
    report = drift_report_from_batch(dummy)
    print(f"[9.7] Drift report generated with {len(report)} features (dummy run).")
except Exception as e:
    print(f"[skip] drift report not generated — {e}")

print(">>> Section 9.7 — Champion Drift Report: complete")


>>> Section 9.7 — Champion Drift Report: start


[skip] drift report not generated — [9.7] Preprocessor not found in staging.
>>> Section 9.7 — Champion Drift Report: complete


## Section 9.8 — Section

In [6]:
# =====================================================
# Section 9.8 — Section
# =====================================================
print(">>> Section 9.8: start")
reports_dir = Path(OUT_ROOT) / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

def _safe_load_json(p: Path):
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            return None
    return None
rows = []
for cand in ['lgbm_report.json', 'lightgbm_report.json', 'champion.json']:
    rep = _safe_load_json(reports_dir / cand)
    if rep:
        ap = rep.get('ap') or rep.get('mean_ap') or rep.get('metrics', {}).get('ap')
        row = {'model': rep.get('model', 'LightGBM (report)'), 'status': rep.get('status', 'ok' if ap is not None else 'pending'), 'ap': float(ap) if ap is not None else None, 'f1_at_tau': rep.get('f1_at_tau'), 'tau_star': rep.get('tau_star'), 'precision': rep.get('precision'), 'recall': rep.get('recall'), 'cm_tn_fp_fn_tp': rep.get('cm_tn_fp_fn_tp'), 'notes': rep.get('reason') or rep.get('notes', '')}
        rows.append(row)
        break
cnn_candidates = ['cnn_report.json', 'cnn_metrics.json', 'cnn_eval.json']
cnn_rep = None
for cand in cnn_candidates:
    tmp = _safe_load_json(reports_dir / cand)
    if tmp:
        cnn_rep = tmp
        break
if cnn_rep:
    ap = cnn_rep.get('ap') or cnn_rep.get('mean_ap') or cnn_rep.get('metrics', {}).get('ap')
    rows.append({'model': cnn_rep.get('model', 'CNN'), 'status': cnn_rep.get('status', 'ok' if ap is not None else 'pending'), 'ap': float(ap) if ap is not None else None, 'f1_at_tau': cnn_rep.get('f1_at_tau'), 'tau_star': cnn_rep.get('tau_star'), 'precision': cnn_rep.get('precision'), 'recall': cnn_rep.get('recall'), 'cm_tn_fp_fn_tp': cnn_rep.get('cm_tn_fp_fn_tp'), 'notes': cnn_rep.get('reason') or cnn_rep.get('notes', '')})
else:
    note = 'CNN metrics unavailable — run Section 2.1 (payload sequence) and Section 7 (CNN training).'
    rows.append({'model': 'CNN', 'status': 'pending', 'ap': None, 'f1_at_tau': None, 'tau_star': None, 'precision': None, 'recall': None, 'cm_tn_fp_fn_tp': None, 'notes': note})
df = pd.DataFrame(rows, columns=['model', 'status', 'ap', 'f1_at_tau', 'tau_star', 'precision', 'recall', 'cm_tn_fp_fn_tp', 'notes'])
display_df = df.copy()
metric_cols = ['ap', 'f1_at_tau', 'tau_star', 'precision', 'recall']
for c in metric_cols:
    mask = (display_df['status'] != 'ok') | display_df[c].isna()
    display_df.loc[mask, c] = '–'
    with np.errstate(all='ignore'):
        display_df.loc[~mask, c] = pd.to_numeric(display_df.loc[~mask, c], errors='coerce').round(6)
df.to_json(reports_dir / 'final_table_raw.json', orient='records', indent=2)
display_df.to_csv(reports_dir / 'final_table.csv', index=False)
display(display_df)
print(f'[ok] final reports written → {reports_dir} (final_table.csv, final_table_raw.json)')

>>> Section 9.8: start


,model,status,ap,f1_at_tau,tau_star,precision,recall,cm_tn_fp_fn_tp,notes
0,LightGBM (report),selected,–,–,–,–,–,None,
1,CNN,pending,–,–,–,–,–,None,CNN metrics unavailable — run Section 2.1 (pay...


[ok] final reports written → out/reports (final_table.csv, final_table_raw.json)


In [7]:
# =====================================================
# Section 9.8 — Section
# =====================================================
print(">>> Section 9.8: start")

>>> Section 9.8: start


In [8]:
# =====================================================
# Section 9.8 — Section
# =====================================================
print(">>> Section 9.8: start")

>>> Section 9.8: start
